<img src="https://img.shields.io/badge/Grafos-Aula%20pr%C3%A1tica-1f4e79?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/NetworkX-3.x-orange?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/OSMnx-2.x-2a78d6?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/Plotly-interativo-9085e9?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/Semana-09-1baf7a?style=for-the-badge" alt="badge">

# ⭐ A-star (A\*) · busca informada, do tabuleiro às ruas de Natal

**Aula prática** · Algoritmos e Estrutura de Dados II (Grafos) · UFRN / DCA · ivanovitch.silva@ufrn.br

> Em 1968, no Stanford Research Institute, um robô chamado **Shakey** precisava atravessar uma sala cheia de caixas sem bater em nada. Ele tinha um mapa, um ponto de partida e um destino. O Dijkstra, publicado nove anos antes, resolvia o problema, mas resolvia **demais**. Ele calculava a melhor rota para todos os cantos da sala antes de responder a única pergunta que importava. Peter Hart, Nils Nilsson e Bertram Raphael perceberam que o robô sabia algo que o Dijkstra ignorava. Sabia **onde ficava o destino**. Essa informação extra, colocada dentro do algoritmo com cuidado matemático, deu origem ao A\*. O asterisco não é enfeite. Na notação deles, $A_1$ e $A_2$ eram versões preliminares, e o $*$ marca a versão **ótima entre todas as que usam a mesma informação**.

Na Semana 07 escrevemos o Dijkstra. Na Semana 08 construímos o min-heap que o torna eficiente. Hoje mudamos **uma única linha**. Em vez de retirar da fila o nó de menor $g(n)$, o custo já pago desde a origem, retiramos o nó de menor

$$f(n) = g(n) + h(n),$$

onde $h(n)$ é uma **estimativa** do custo que ainda falta até o destino. Uma soma muda tudo. O resto deste notebook mostra o quanto.

## 🎯 O que você vai conseguir fazer ao final

1. Implementar o A\* do zero com `heapq` e explicar cada decisão de projeto (fila de prioridade, remoção preguiçosa, conjunto fechado, parada ao expandir o destino).
2. Definir heurística **admissível** e **consistente**, demonstrar por que a primeira garante o ótimo e o que a segunda acrescenta.
3. Ler a **pegada de busca** de um algoritmo (quais nós ele expandiu e em que ordem) e explicar por que a do Dijkstra é um disco, a do A\* é uma elipse e a do guloso é um feixe.
4. Percorrer o espectro $f = g + w \cdot h$, do Dijkstra ($w = 0$) à busca gulosa ($w \to \infty$), e enunciar a garantia de subotimalidade limitada do A\* ponderado.
5. Diagnosticar um erro de **unidades** em uma heurística usando apenas os resultados de uma tabela de desempenho.
6. Medir desempenho da forma certa, separando nós expandidos de tempo de parede, e explicar por que um A\* pode ser mais lento que o Dijkstra mesmo expandindo menos nós.



# 1 · Preparação do ambiente

Duas bibliotecas além do que já usamos no curso. O **OSMnx** baixa e modela malhas viárias do OpenStreetMap como grafos do NetworkX (com comprimento, velocidade e tempo de viagem em cada aresta). O **Plotly** faz as figuras interativas e animadas. Os widgets (`ipywidgets`) já vêm no Colab. A célula abaixo instala apenas o que faltar.

In [ ]:
# Instala o que faltar sem poluir a saída. No Colab, OSMnx não vem pré-instalado
# e leva cerca de um minuto na primeira execução.
import importlib, subprocess, sys

def garantir(pacote, modulo=None):
    """Importa `modulo` (ou `pacote`) e, se não existir, instala via pip em silêncio."""
    nome = modulo or pacote
    try:
        importlib.import_module(nome)
    except ImportError:
        print(f"Instalando {pacote} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=True)

for pacote, modulo in [("osmnx", None), ("plotly", None), ("ipywidgets", None), ("scikit-learn", "sklearn")]:
    garantir(pacote, modulo)
print("Ambiente pronto.")

In [ ]:
import heapq, math, time, timeit, itertools, random, collections, dataclasses, warnings
import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
print("OSMnx", ox.__version__, "| NetworkX", nx.__version__, "| Plotly", __import__("plotly").__version__)

## 1.1 · Paleta e estúdio visual

Antes de qualquer algoritmo, as cores. Decidir isso uma única vez e no começo é o que permite ler as figuras do notebook inteiro sem legenda. O dicionário `COR` é a nossa constituição visual, e as funções `tonalidade` e `mapa_de_cores` derivam dela os tons claros (fronteira, nós fechados) e os gradientes (ordem de expansão).

In [ ]:
# Uma cor por algoritmo, usada em TODAS as figuras do notebook (princípio da similaridade).
# Quem vê azul já sabe que é Dijkstra, sem ler a legenda.
COR = {
    "Dijkstra":     "#2a78d6",   # azul
    "Guloso":       "#eb6834",   # laranja
    "A*":           "#1baf7a",   # verde
    "A* ponderado": "#9085e9",   # roxo
    "fronteira":    "#f2b705",   # amarelo (nós descobertos, ainda não expandidos)
    "obstaculo":    "#2b2d42",   # grafite
    "livre":        "#f7f7f5",   # quase branco
    "origem":       "#111111",   # preto
    "destino":      "#e5484d",   # vermelho
    "fundo_mapa":   "#0f1117",   # fundo escuro dos mapas (figura e fundo)
    "ruas":         "#3a3f4b",   # cinza das ruas no fundo escuro
}

def tonalidade(cor_hex, fator=0.35):
    """Mistura `cor_hex` com branco. fator=1 devolve a cor, fator→0 tende ao branco."""
    r, g, b = mcolors.to_rgb(cor_hex)
    return mcolors.to_hex((1 - fator + fator * r, 1 - fator + fator * g, 1 - fator + fator * b))

def mapa_de_cores(cor_hex):
    """Gradiente do tom claro ao tom cheio de uma cor, para pintar a ordem de expansão."""
    return mcolors.LinearSegmentedColormap.from_list("grad", [tonalidade(cor_hex, 0.15), cor_hex])

LAYOUT_CLARO = dict(template="plotly_white", font=dict(family="Inter, Arial", size=13),
                    margin=dict(l=40, r=20, t=60, b=40))

# 2 · O mecanismo em uma tela

Um mapa de cidade tem dezenas de milhares de nós e nenhum deles cabe na sua retina. Para **ver** o algoritmo, começamos em um tabuleiro de 20 × 30 células com obstáculos, onde cada passo é visível. O todo antes das partes. Quando chegarmos a Natal, você já vai saber o que procurar.

Três conceitos organizam tudo o que vem a seguir.

| Símbolo | Nome | O que é | Quem calcula |
|---|---|---|---|
| $g(n)$ | custo acumulado | o menor custo **já conhecido** da origem até $n$ | o algoritmo, somando pesos das arestas |
| $h(n)$ | heurística | uma **estimativa** do custo de $n$ até o destino | você, antes de rodar, com conhecimento do problema |
| $f(n) = g(n) + h(n)$ | prioridade | o custo total **estimado** de um caminho que passa por $n$ | a soma dos dois |

O A\* mantém dois conjuntos. A **fronteira** (ou conjunto aberto) guarda os nós já descobertos e ainda não examinados, organizados em uma fila de prioridade pelo menor $f$. O conjunto **fechado** guarda os nós já expandidos. Expandir um nó significa retirá-lo da fronteira e olhar seus vizinhos. A busca termina quando o **destino é expandido**, não quando é descoberto. Essa distinção parece miúda e é o que garante o ótimo.

## 2.1 · Um tabuleiro como grafo

Um tabuleiro é um grafo em grade. `nx.grid_2d_graph(n, m)` cria os nós $(linha, coluna)$ e as arestas entre vizinhos ortogonais. Remover os nós sorteados como obstáculos é tudo o que falta. Guardamos a máscara de obstáculos, a origem e o destino em `G.graph` para as funções de desenho.

In [ ]:
def criar_tabuleiro(n_linhas=20, n_colunas=30, densidade=0.28, semente=13,
                    origem=None, destino=None, diagonais=False):
    """Cria um tabuleiro com obstáculos aleatórios como um grafo NetworkX.

    Cada célula livre é um nó (linha, coluna). Células vizinhas (4 direções, ou 8 se
    `diagonais=True`) são ligadas por arestas de peso 1 (ou √2 nas diagonais).
    Os obstáculos são removidos do grafo. A semente é incrementada até existir um
    caminho entre origem e destino, para o experimento nunca cair em um tabuleiro
    sem solução.

    Retorna um nx.Graph com metadados em G.graph (dimensões, máscara de obstáculos,
    origem e destino).
    """
    origem = origem or (n_linhas // 2, 1)
    destino = destino or (n_linhas // 2, n_colunas - 2)
    while True:
        rng = np.random.default_rng(semente)
        obst = rng.random((n_linhas, n_colunas)) < densidade
        obst[origem] = obst[destino] = False
        G = nx.grid_2d_graph(n_linhas, n_colunas)
        if diagonais:
            for r in range(n_linhas):
                for c in range(n_colunas):
                    for dr, dc in [(1, 1), (1, -1)]:
                        if 0 <= r + dr < n_linhas and 0 <= c + dc < n_colunas:
                            G.add_edge((r, c), (r + dr, c + dc), weight=math.sqrt(2))
        nx.set_edge_attributes(G, {e: 1.0 for e in G.edges if "weight" not in G.edges[e]}, "weight")
        G.remove_nodes_from([(r, c) for r in range(n_linhas) for c in range(n_colunas) if obst[r, c]])
        if nx.has_path(G, origem, destino):
            break
        semente += 1
    G.graph.update(n_linhas=n_linhas, n_colunas=n_colunas, obstaculos=obst,
                   origem=origem, destino=destino, semente=semente)
    return G


def tabuleiro_armadilha(n_linhas=21, n_colunas=35, profundidade=8, meia_altura=5):
    """Tabuleiro com uma cavidade em forma de C aberta PARA a origem.

    É a armadilha clássica da busca gulosa. A linha reta da origem ao destino entra
    na cavidade, que não tem saída, e a heurística continua dizendo "siga em frente".
    """
    obst = np.zeros((n_linhas, n_colunas), dtype=bool)
    meio = n_linhas // 2
    origem = (meio, 2)
    destino = (meio, n_colunas - 4)
    fundo = n_colunas - 12                                 # parede do fundo da cavidade
    topo, base = meio - meia_altura, meio + meia_altura
    obst[topo:base + 1, fundo] = True                      # fundo (vertical)
    obst[topo, fundo - profundidade:fundo + 1] = True      # braço superior, apontando para a origem
    obst[base, fundo - profundidade:fundo + 1] = True      # braço inferior
    G = nx.grid_2d_graph(n_linhas, n_colunas)
    nx.set_edge_attributes(G, 1.0, "weight")
    G.remove_nodes_from([(r, c) for r in range(n_linhas) for c in range(n_colunas) if obst[r, c]])
    assert nx.has_path(G, origem, destino), "a cavidade fechou o destino"
    G.graph.update(n_linhas=n_linhas, n_colunas=n_colunas, obstaculos=obst,
                   origem=origem, destino=destino, semente=None)
    return G


G_tab = criar_tabuleiro()
print(f"{G_tab.number_of_nodes()} células livres, {G_tab.number_of_edges()} arestas, "
      f"origem {G_tab.graph['origem']} → destino {G_tab.graph['destino']}")

## 2.2 · Heurísticas para o tabuleiro

Em uma grade onde só se anda nas quatro direções, com custo 1 por passo, o menor número de passos entre duas células **sem obstáculos** é a distância de Manhattan. Com obstáculos, o caminho real só pode ser maior. Logo Manhattan nunca superestima. É admissível e, nesse cenário, é a heurística admissível mais apertada que existe.

| Heurística | Fórmula | Admissível em grade 4-conexa? | Comentário |
|---|---|---|---|
| zero | $0$ | sim | A\* vira Dijkstra |
| Manhattan | $\lvert \Delta r \rvert + \lvert \Delta c \rvert$ | sim, e exata sem obstáculos | a melhor para 4 direções |
| Euclidiana | $\sqrt{\Delta r^2 + \Delta c^2}$ | sim | subestima mais que Manhattan, logo expande mais nós |
| Chebyshev | $\max(\lvert \Delta r \rvert, \lvert \Delta c \rvert)$ | sim | exata em grade 8-conexa com diagonal de custo 1 |
| Manhattan × 2 | $2(\lvert \Delta r \rvert + \lvert \Delta c \rvert)$ | **não** | superestima, pode perder o ótimo |

Repare na ordem. Para o mesmo problema, $0 \le \text{Chebyshev} \le \text{Euclidiana} \le \text{Manhattan} \le h^*$. Quanto mais perto de $h^*$ (o custo real restante), menos nós o A\* expande. Uma heurística admissível que **domina** outra nunca expande mais nós do que ela.

In [ ]:
# Heurísticas para o tabuleiro. Os nós são tuplas (linha, coluna).
def h_zero(u, v):
    """Sem informação. A* com h = 0 é exatamente o Dijkstra."""
    return 0.0

def h_manhattan(u, v):
    """Distância em quarteirões. Exata quando só há movimentos nas 4 direções."""
    return abs(u[0] - v[0]) + abs(u[1] - v[1])

def h_euclidiana(u, v):
    """Distância em linha reta. Admissível, mas subestima mais que Manhattan em grades 4-conexas."""
    return math.hypot(u[0] - v[0], u[1] - v[1])

def h_chebyshev(u, v):
    """Número de passos de rei no xadrez. Exata em grades 8-conexas com diagonal de custo 1."""
    return max(abs(u[0] - v[0]), abs(u[1] - v[1]))

def h_octil(u, v):
    """Distância em grade 8-conexa com diagonal de custo √2."""
    dx, dy = abs(u[0] - v[0]), abs(u[1] - v[1])
    return max(dx, dy) + (math.sqrt(2) - 1) * min(dx, dy)

def escalar(h, fator):
    """Multiplica uma heurística por um fator. fator > 1 pode torná-la inadmissível."""
    return lambda u, v: fator * h(u, v)

HEURISTICAS_GRADE = {
    "zero (Dijkstra)": h_zero,
    "Manhattan": h_manhattan,
    "Euclidiana": h_euclidiana,
    "Chebyshev": h_chebyshev,
    "Manhattan × 2 (inadmissível)": escalar(h_manhattan, 2.0),
}

## 2.3 · O A\* do zero, instrumentado

A função abaixo é o coração do notebook. Ela é o pseudocódigo dos slides escrito em Python, com seis decisões que merecem comentário.

1. **A fila de prioridade é o `heapq`**, o mesmo min-heap que construímos na Semana 08, agora pronto na biblioteca padrão. Inserir e retirar custam $O(\log n)$.
2. **Remoção preguiçosa.** Quando descobrimos um caminho melhor para um nó que já está na fila, não procuramos a entrada antiga para apagá-la (o heap não suporta isso em $O(\log n)$). Inserimos uma nova entrada com $f$ menor e, quando a antiga sair, ela é ignorada porque o nó já está fechado. O NetworkX faz exatamente isso em `nx.astar_path`.
3. **O contador `ordem`** desempata prioridades iguais pela ordem de chegada e impede que o `heapq` tente comparar dois nós entre si (tuplas são comparadas elemento a elemento, e nós podem não ser comparáveis).
4. **Parar ao expandir o destino, não ao descobri-lo.** Quando o destino sai da fila com o menor $f$ de toda a fronteira, nenhum outro caminho pode ser mais barato (desde que $h$ seja admissível). Parar ao descobri-lo abriria mão dessa garantia.
5. **Conjunto fechado sem reabertura.** Um nó fechado não volta à fila. Isso só é correto quando a heurística é **consistente** (seção 3). O parâmetro `reabrir=True` liga a reabertura para heurísticas que não são.
6. **O peso $w$** em $f = g + w \cdot h$ coloca Dijkstra ($w = 0$), A\* ($w = 1$), A\* ponderado ($w > 1$) e busca gulosa ($w = \infty$, só $h$) na mesma função. A mesma máquina, quatro comportamentos.

A instrumentação registra **em que passo cada nó entrou na fronteira e em que passo saiu dela**. Com dois dicionários reconstruímos o estado da busca em qualquer instante sem guardar uma cópia por passo. É o que alimenta as animações.

In [ ]:
@dataclasses.dataclass
class Busca:
    """Tudo o que uma execução da busca deixa para trás, além do caminho.

    Os dicionários `descoberto_em` e `expandido_em` guardam em que passo cada nó
    entrou na fronteira e em que passo saiu dela. Com eles reconstruímos o estado
    da busca em qualquer instante, sem guardar uma cópia por passo.
    """
    nome: str
    caminho: list
    custo: float
    expandidos: list                 # nós na ordem em que saíram da fila
    g: dict                          # melhor custo conhecido da origem até cada nó
    f: dict                          # prioridade com que cada nó entrou na fila pela última vez
    pai: dict                        # para reconstruir o caminho
    descoberto_em: dict              # passo em que o nó entrou na fronteira
    expandido_em: dict               # passo em que o nó foi expandido
    n_relaxamentos: int              # quantas arestas foram relaxadas
    tempo_s: float                   # tempo de parede desta execução

    @property
    def n_expandidos(self):
        return len(self.expandidos)


def reconstruir_caminho(pai, atual):
    """Volta pelos ponteiros `pai` do destino até a origem e inverte a lista."""
    caminho = [atual]
    while atual in pai:
        atual = pai[atual]
        caminho.append(atual)
    caminho.reverse()
    return caminho


def a_star(G, origem, destino, heuristica=None, peso="weight", w=1.0,
           reabrir=False, nome=None):
    """A* instrumentado sobre um grafo NetworkX (simples ou multigrafo, dirigido ou não).

    Parâmetros
    ----------
    heuristica : função (u, destino) -> estimativa do custo restante. `None` equivale a h = 0,
                 ou seja, Dijkstra.
    peso       : nome do atributo de aresta usado como custo (ex.: "weight", "length", "travel_time").
    w          : peso da heurística em f = g + w·h. w = 0 é Dijkstra, w = 1 é o A* clássico,
                 w = math.inf ignora g e vira a busca gulosa (f = h).
    reabrir    : se True, um nó já fechado pode voltar à fila quando um caminho melhor aparece.
                 Só é necessário quando a heurística não é consistente.

    A fila de prioridade é o `heapq` da biblioteca padrão (o mesmo min-heap da Semana 08).
    Entradas obsoletas não são removidas da fila, apenas ignoradas quando saem
    ("remoção preguiçosa"). O contador `ordem` desempata prioridades iguais pela
    ordem de chegada e evita comparar nós entre si.
    """
    inicio = time.perf_counter()
    multigrafo = G.is_multigraph()

    def vizinhos(u):
        # Em um multigrafo (como os do OSMnx) pode haver mais de uma aresta u→v.
        # Para a busca só importa a mais barata.
        for v, dados in G.adj[u].items():
            if multigrafo:
                yield v, min(d.get(peso, 1.0) for d in dados.values())
            else:
                yield v, dados.get(peso, 1.0)

    if heuristica is None or w == 0:
        h = lambda n: 0.0
    else:
        h = lambda n: heuristica(n, destino)

    def prioridade(g_n, h_n):
        return h_n if w == math.inf else g_n + w * h_n

    ordem = itertools.count()
    g = {origem: 0.0}
    f = {origem: prioridade(0.0, h(origem))}
    pai, fechados = {}, set()
    expandidos, descoberto_em, expandido_em = [], {origem: 0}, {}
    fila = [(f[origem], next(ordem), origem)]
    passo, n_relax = 0, 0

    while fila:
        _, _, u = heapq.heappop(fila)
        if u in fechados:
            continue                               # entrada obsoleta da fila
        fechados.add(u)
        passo += 1
        expandidos.append(u)
        expandido_em[u] = passo
        if u == destino:
            break                                  # parar ao EXPANDIR o destino, não ao descobri-lo
        for v, custo in vizinhos(u):
            g_tentativo = g[u] + custo
            if v in fechados:
                if not (reabrir and g_tentativo < g[v]):
                    continue
                fechados.discard(v)                # reabertura (heurística inconsistente)
            if g_tentativo < g.get(v, math.inf):
                g[v] = g_tentativo
                pai[v] = u
                f[v] = prioridade(g_tentativo, h(v))
                heapq.heappush(fila, (f[v], next(ordem), v))
                descoberto_em.setdefault(v, passo)
                n_relax += 1

    caminho = reconstruir_caminho(pai, destino) if destino in g and destino in fechados else []
    custo = g[destino] if caminho else math.inf
    if nome is None:
        nome = "Dijkstra" if heuristica is None or w == 0 else ("Guloso" if w == math.inf else ("A*" if w == 1 else f"A* (w={w})"))
    return Busca(nome, caminho, custo, expandidos, g, f, pai, descoberto_em, expandido_em,
                 n_relax, time.perf_counter() - inicio)

## 2.4 · Três buscas no mesmo tabuleiro

Dijkstra, A\* com Manhattan e guloso com Manhattan, no mesmo tabuleiro e no mesmo par origem-destino. A referência de corretude é o NetworkX, que precisa concordar com o nosso custo ótimo. Antes de rodar, aposte. Qual expande menos nós? Qual encontra o caminho mais curto?

In [ ]:
# Três buscas sobre o mesmo tabuleiro. A referência de corretude é o próprio NetworkX.
o, d = G_tab.graph["origem"], G_tab.graph["destino"]
b_dij = a_star(G_tab, o, d)                                  # h = 0
b_ast = a_star(G_tab, o, d, heuristica=h_manhattan)          # A* clássico
b_gul = a_star(G_tab, o, d, heuristica=h_manhattan, w=math.inf)  # guloso

custo_nx = nx.shortest_path_length(G_tab, o, d, weight="weight")
assert b_dij.custo == custo_nx == b_ast.custo, "A* com heurística admissível tem que achar o ótimo"
assert b_ast.caminho == nx.astar_path(G_tab, o, d, heuristic=h_manhattan, weight="weight") or b_ast.custo == custo_nx

for b in (b_dij, b_ast, b_gul):
    print(f"{b.nome:10s} custo = {b.custo:5.1f} | nós expandidos = {b.n_expandidos:4d} | "
          f"relaxamentos = {b.n_relaxamentos:4d} | {b.tempo_s*1000:.2f} ms")

Dois resultados previsíveis e um que surpreende quem vê pela primeira vez. Dijkstra e A\* encontram **o mesmo custo** (ambos ótimos), e o A\* chega lá expandindo uma fração dos nós. O guloso expande ainda menos e **paga por isso com um caminho pior**. Guarde esses três números. O notebook inteiro é uma variação sobre eles.

## 2.5 · Passo a passo

Agora o mecanismo em câmera lenta. Cada célula do tabuleiro tem um estado, e cada estado tem uma cor fixa.

| Estado | Cor | Significado |
|---|---|---|
| livre | quase branco | ainda não vista |
| obstáculo | grafite | não é nó do grafo |
| fronteira | amarelo | descoberta, na fila, aguardando |
| fechado | tom claro da cor do algoritmo | expandida |
| caminho | cor cheia do algoritmo | aparece no último quadro |
| origem e destino | preto e vermelho | fixos |

Passe o mouse sobre uma célula para ver $g$, $f$ e os passos em que ela entrou e saiu da fronteira. Use o slider para ir e voltar, e repare na fronteira amarela. Ela é a **frente de onda** da busca.

In [ ]:
# Estados de uma célula, do fundo para a figura.
LIVRE, OBSTACULO, FECHADO, FRONTEIRA, CAMINHO, ORIGEM, DESTINO = range(7)

def estado_no_passo(G, busca, k):
    """Matriz de estados do tabuleiro após `k` expansões da busca.

    k = 0 mostra só o tabuleiro; k >= número de expansões mostra também o caminho.
    """
    n, m = G.graph["n_linhas"], G.graph["n_colunas"]
    M = np.full((n, m), LIVRE, dtype=int)
    M[G.graph["obstaculos"]] = OBSTACULO
    for no, p in busca.descoberto_em.items():
        if p <= k:
            M[no] = FRONTEIRA
    for no, p in busca.expandido_em.items():
        if p <= k:
            M[no] = FECHADO
    if k >= busca.n_expandidos:
        for no in busca.caminho:
            M[no] = CAMINHO
    M[G.graph["origem"]] = ORIGEM
    M[G.graph["destino"]] = DESTINO
    return M


def escala_discreta(cor_algoritmo):
    """Escala de cores do Plotly com uma cor fixa por estado (fechado e caminho na cor do algoritmo)."""
    cores = [COR["livre"], COR["obstaculo"], tonalidade(cor_algoritmo, 0.45),
             COR["fronteira"], cor_algoritmo, COR["origem"], COR["destino"]]
    n = len(cores)
    escala = []
    for i, c in enumerate(cores):
        escala += [[i / n, c], [(i + 1) / n, c]]
    return escala


def texto_hover(G, busca):
    """Texto de hover por célula com g, h e f finais (ou 'não visitado')."""
    n, m = G.graph["n_linhas"], G.graph["n_colunas"]
    T = np.empty((n, m), dtype=object)
    for r in range(n):
        for c in range(m):
            no = (r, c)
            if G.graph["obstaculos"][r, c]:
                T[r, c] = "obstáculo"
            elif no in busca.g:
                f_val = busca.f.get(no, float("nan"))
                T[r, c] = (f"({r}, {c})<br>g = {busca.g[no]:.1f}<br>f = {f_val:.1f}<br>"
                           f"descoberto no passo {busca.descoberto_em.get(no, '-')}<br>"
                           f"expandido no passo {busca.expandido_em.get(no, '-')}")
            else:
                T[r, c] = f"({r}, {c})<br>não visitado"
    return T

In [ ]:
def heatmap_estado(M, T, cor):
    """Um go.Heatmap para uma matriz de estados."""
    return go.Heatmap(z=M, text=T, hovertemplate="%{text}<extra></extra>",
                      colorscale=escala_discreta(cor), zmin=-0.5, zmax=6.5,
                      showscale=False, xgap=1, ygap=1)


def animar_busca(G, busca, cor=None, passo=2, titulo=None, altura=520):
    """Animação passo a passo de UMA busca no tabuleiro, com botão de play e slider.

    `passo` controla quantas expansões avançam por quadro.
    """
    cor = cor or COR.get(busca.nome.split(" (")[0], COR["A*"])
    T = texto_hover(G, busca)
    quadros_k = list(range(0, busca.n_expandidos + 1, passo))
    if quadros_k[-1] != busca.n_expandidos:
        quadros_k.append(busca.n_expandidos)
    fig = go.Figure(
        data=[heatmap_estado(estado_no_passo(G, busca, 0), T, cor)],
        frames=[go.Frame(data=[heatmap_estado(estado_no_passo(G, busca, k), T, cor)],
                         name=str(k)) for k in quadros_k],
    )
    passos_slider = [dict(method="animate", label=str(k),
                          args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                              transition=dict(duration=0))]) for k in quadros_k]
    fig.update_layout(
        title=titulo or f"{busca.nome} · {busca.n_expandidos} expansões · custo {busca.custo:.0f}",
        height=altura, margin=dict(l=20, r=20, t=60, b=20), template="plotly_white",
        xaxis=dict(visible=False, constrain="domain"),
        yaxis=dict(visible=False, autorange="reversed", scaleanchor="x", scaleratio=1),
        updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=-0.02, xanchor="left", yanchor="top",
                          buttons=[dict(label="▶ play", method="animate",
                                        args=[None, dict(frame=dict(duration=60, redraw=True),
                                                         fromcurrent=True, transition=dict(duration=0))]),
                                   dict(label="❚❚ pausa", method="animate",
                                        args=[[None], dict(frame=dict(duration=0, redraw=False),
                                                           mode="immediate")])])],
        sliders=[dict(active=0, steps=passos_slider, x=0.15, len=0.85, y=-0.02, yanchor="top",
                      currentvalue=dict(prefix="expansões: ", visible=True))],
    )
    return fig

In [ ]:
animar_busca(G_tab, b_ast).show()

## 2.6 · Destino comum

Três painéis, o mesmo tabuleiro, as três buscas avançando **no mesmo ritmo** (mesmo número de expansões por quadro). Quando uma termina, seu painel congela com o caminho e os outros continuam. Aperte play e não conte nada. Só olhe quem para primeiro, quem para por último e o formato da mancha que cada um deixa.

In [ ]:
def animar_tres(G, buscas, passo=3, altura=430, titulo="Três buscas, um destino"):
    """Três buscas lado a lado avançando no MESMO ritmo (mesmo número de expansões por quadro).

    Quando uma busca termina, seu painel congela com o caminho enquanto as outras
    continuam. É o princípio do destino comum em ação. O que se move junto é lido
    como um só processo; o que para primeiro salta aos olhos.
    """
    nomes = list(buscas)
    cores = [COR.get(n.split(" (")[0], COR["A*"]) for n in nomes]
    textos = [texto_hover(G, buscas[n]) for n in nomes]
    k_max = max(b.n_expandidos for b in buscas.values())
    quadros_k = list(range(0, k_max + 1, passo))
    if quadros_k[-1] != k_max:
        quadros_k.append(k_max)

    def subtitulo(n, k):
        b = buscas[n]
        if k >= b.n_expandidos:
            return f"{n} · {b.n_expandidos} expansões · custo {b.custo:.0f} ✓"
        return f"{n} · {min(k, b.n_expandidos)} expansões"

    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.03,
                        subplot_titles=[subtitulo(n, 0) for n in nomes])
    for i, n in enumerate(nomes):
        fig.add_trace(heatmap_estado(estado_no_passo(G, buscas[n], 0), textos[i], cores[i]), row=1, col=i + 1)
        fig.update_xaxes(visible=False, row=1, col=i + 1)
        fig.update_yaxes(visible=False, autorange="reversed", scaleanchor=f"x{'' if i == 0 else i + 1}",
                         scaleratio=1, row=1, col=i + 1)

    frames = []
    for k in quadros_k:
        anotacoes = [a.to_plotly_json() for a in fig.layout.annotations]
        for i, n in enumerate(nomes):
            anotacoes[i]["text"] = subtitulo(n, k)
        layout_k = go.Layout(annotations=anotacoes)
        frames.append(go.Frame(name=str(k), traces=[0, 1, 2], layout=layout_k,
                               data=[heatmap_estado(estado_no_passo(G, buscas[n], k), textos[i], cores[i])
                                     for i, n in enumerate(nomes)]))
    fig.frames = frames
    passos_slider = [dict(method="animate", label=str(k),
                          args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                              transition=dict(duration=0))]) for k in quadros_k]
    fig.update_layout(
        title=titulo, height=altura, template="plotly_white", margin=dict(l=10, r=10, t=70, b=20),
        updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=-0.03, xanchor="left", yanchor="top",
                          buttons=[dict(label="▶ play", method="animate",
                                        args=[None, dict(frame=dict(duration=70, redraw=True),
                                                         fromcurrent=True, transition=dict(duration=0))]),
                                   dict(label="❚❚ pausa", method="animate",
                                        args=[[None], dict(frame=dict(duration=0, redraw=False),
                                                           mode="immediate")])])],
        sliders=[dict(active=0, steps=passos_slider, x=0.15, len=0.85, y=-0.03, yanchor="top",
                      currentvalue=dict(prefix="expansões: ", visible=True))],
    )
    return fig

In [ ]:
animar_tres(G_tab, {"Dijkstra": b_dij, "A*": b_ast, "Guloso": b_gul}).show()

O Dijkstra pinta um **disco** em volta da origem, porque expande por ordem de $g$ e o conjunto $\{n : g(n) \le C\}$ é uma bola de raio $C$ na métrica do grafo. O A\* pinta uma **elipse alongada na direção do destino**. O guloso pinta um **feixe fino** que às vezes erra. Na seção 5.4 isso vira teorema.

## 2.7 · A armadilha do guloso

A heurística diz "o destino está ali". A busca gulosa acredita e segue em linha reta. Quando a linha reta termina em uma parede em forma de C, ela entra na cavidade, enche a cavidade inteira (porque cada célula lá dentro parece mais perto do destino que qualquer célula fora) e só depois dá a volta. O A\* também entra, mas o $g$ crescente faz cada passo dentro da cavidade ficar mais caro, e ele desiste cedo. O Dijkstra nem olha para o destino.

Os desenhos abaixo são estáticos (Matplotlib). A versão animada vem em seguida.

In [ ]:
def desenhar_tabuleiro(G, busca, ax=None, cor=None, titulo=None, mostrar_ordem=False):
    """Versão estática (Matplotlib) do estado final de uma busca. Rápida, serve para os widgets."""
    cor = cor or COR.get(busca.nome.split(" (")[0], COR["A*"])
    M = estado_no_passo(G, busca, busca.n_expandidos)
    cores = [COR["livre"], COR["obstaculo"], tonalidade(cor, 0.45), COR["fronteira"],
             cor, COR["origem"], COR["destino"]]
    cmap = mcolors.ListedColormap(cores)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7 * G.graph["n_linhas"] / G.graph["n_colunas"]))
    ax.imshow(M, cmap=cmap, vmin=-0.5, vmax=6.5, interpolation="nearest")
    if mostrar_ordem:
        for no, p in busca.expandido_em.items():
            ax.text(no[1], no[0], str(p), ha="center", va="center", fontsize=6, color="#222")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(titulo or f"{busca.nome} · {busca.n_expandidos} expansões · custo {busca.custo:.0f}",
                 fontsize=11, loc="left")
    return ax

In [ ]:
G_arm = tabuleiro_armadilha()
o, d = G_arm.graph["origem"], G_arm.graph["destino"]
buscas_arm = {"Dijkstra": a_star(G_arm, o, d),
              "A*": a_star(G_arm, o, d, heuristica=h_manhattan),
              "Guloso": a_star(G_arm, o, d, heuristica=h_manhattan, w=math.inf)}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, (nome, b) in zip(axes, buscas_arm.items()):
    desenhar_tabuleiro(G_arm, b, ax=ax, cor=COR[nome])
plt.tight_layout()
plt.show()

In [ ]:
animar_tres(G_arm, buscas_arm, passo=6, titulo="A armadilha, passo a passo").show()

# 3 · Heurísticas e admissibilidade

Agora a teoria que sustenta o que você acabou de ver. Seja $h^*(n)$ o custo **real** do caminho ótimo de $n$ até o destino.

**Definição (admissibilidade).** Uma heurística $h$ é admissível quando nunca superestima, isto é, $h(n) \le h^*(n)$ para todo nó $n$. Em particular, $h(\text{destino}) = 0$.

**Teorema (Hart, Nilsson e Raphael, 1968).** Se $h$ é admissível, o A\* devolve um caminho de custo ótimo $C^*$.

*Esboço da prova.* Suponha que o A\* expandiu o destino por um caminho de custo $C > C^*$. Existe um caminho ótimo da origem ao destino, e nele há um primeiro nó $m$ que ainda estava na fronteira no momento da expansão do destino. Como $m$ está no caminho ótimo, $g(m)$ é o custo ótimo até $m$, e por admissibilidade $f(m) = g(m) + h(m) \le g(m) + h^*(m) = C^*$. Mas o destino saiu da fila com $f = C > C^* \ge f(m)$, contradizendo a escolha do menor $f$. $\square$

Repare que a prova usa a parada **ao expandir** o destino. É a decisão 4 da seção 2.3.

**Definição (consistência, ou monotonicidade).** $h$ é consistente quando, para toda aresta $(n, n')$ de custo $c(n, n')$, vale $h(n) \le c(n, n') + h(n')$. É a desigualdade triangular aplicada à estimativa. Toda heurística consistente é admissível (a recíproca é falsa, embora na prática quase toda heurística admissível natural também seja consistente).

**O que a consistência compra.** Com $h$ consistente, os valores de $f$ ao longo de qualquer caminho nunca diminuem e, portanto, **a sequência de $f$ dos nós expandidos é não decrescente**. Consequência prática. Quando um nó é expandido pela primeira vez, seu $g$ já é ótimo. Ele nunca precisa ser reaberto, e o conjunto fechado pode ser um simples `set`. É por isso que a nossa implementação usa `reabrir=False` por padrão. Com heurística admissível mas inconsistente, o A\* ainda acha o ótimo, mas só se permitir reabrir nós fechados, e o número de expansões pode explodir.

**Dominância.** Se $h_1$ e $h_2$ são consistentes e $h_2(n) \ge h_1(n)$ para todo $n$, então o A\* com $h_2$ expande um subconjunto dos nós que expande com $h_1$ (a menos de empates). Heurística mais informada, pegada menor. É a ordem $0 \le \text{Chebyshev} \le \text{Euclidiana} \le \text{Manhattan}$ da seção 2.2.

**A\* ponderado (Pohl, 1970).** Com $f = g + w \cdot h$ e $w > 1$, a heurística efetiva $w \cdot h$ pode superestimar por um fator no máximo $w$. O caminho devolvido custa **no máximo $w \cdot C^*$**. Essa é a garantia de **subotimalidade limitada**. Em troca, a busca expande bem menos nós. O guloso é o limite $w \to \infty$, sem garantia nenhuma.

**Quantos nós o A\* expande?** Todo nó com $f(n) < C^*$ é expandido (alguns com $f(n) = C^*$ também, dependendo dos empates). Esse é o conjunto que você viu pintado. Quanto melhor $h$, menor esse conjunto. Na pior hipótese ($h = 0$) ele é o do Dijkstra. O custo por expansão é $O(\log V)$ no heap mais o custo de **chamar $h$** para cada vizinho, e esse segundo termo vai nos surpreender na seção 6.

## 3.1 · Comparando heurísticas no tabuleiro

As cinco heurísticas da seção 2.2 no tabuleiro padrão. A coluna `h_na_origem` mostra a estimativa inicial. Veja a ordem de dominância aparecer na coluna de nós expandidos e a inadmissível perder o ótimo.

In [ ]:
def estilizar(df):
    """Tabela com gradientes para leitura rápida (nós expandidos em azul, desvio do ótimo em vermelho).
    Espera as colunas nos_expandidos, custo_vs_otimo e otimo."""
    fmt = {"tempo_ms": "{:.1f}", "desvio_ms": "{:.1f}", "nos_expandidos": "{:,}", "custo": "{:,.1f}",
           "distancia_m": "{:,.0f}", "tempo_rota_s": "{:,.0f}", "custo_vs_otimo": "{:.3f}"}
    fmt = {k: v for k, v in fmt.items() if k in df.columns}
    sty = (df.style.format(fmt)
             .background_gradient(subset=["nos_expandidos"], cmap="Blues")
             .background_gradient(subset=["custo_vs_otimo"], cmap="Reds", vmin=1.0, vmax=1.5)
             .map(lambda v: "color: #1baf7a; font-weight: 600" if v is True or v == True else "color: #e5484d; font-weight: 600",
                  subset=["otimo"]))
    return sty

In [ ]:
o, d = G_tab.graph["origem"], G_tab.graph["destino"]
linhas = []
for nome, h in HEURISTICAS_GRADE.items():
    b = a_star(G_tab, o, d, heuristica=h, nome=nome)
    linhas.append(dict(heuristica=nome, h_na_origem=h(o, d), nos_expandidos=b.n_expandidos, custo=b.custo))
tabela_grade = pd.DataFrame(linhas)
tabela_grade["otimo"] = np.isclose(tabela_grade["custo"], tabela_grade["custo"].min())
tabela_grade["custo_vs_otimo"] = tabela_grade["custo"] / tabela_grade["custo"].min()
estilizar(tabela_grade).format({"h_na_origem": "{:.2f}"})

## 3.2 · O espectro do A\* ponderado

Variamos $w$ de 0 a $\infty$ com a heurística de Manhattan. As barras são os nós expandidos (azul em $w=0$, verde em $w=1$, roxo entre, laranja no guloso) e a linha vermelha é o custo do caminho dividido pelo ótimo. Até $w = 1$ a linha fica em 1 (garantia do teorema). Depois de 1 ela pode subir, mas nunca acima de $w$ (garantia do A\* ponderado).

In [ ]:
def espectro_w(G, heuristica, pesos=(0, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 3, 5, math.inf)):
    """Varre o peso w em f = g + w·h e registra nós expandidos e custo do caminho."""
    o, d = G.graph["origem"], G.graph["destino"]
    linhas = []
    for w in pesos:
        b = a_star(G, o, d, heuristica=heuristica, w=w)
        linhas.append(dict(w=w, rotulo=("∞ (guloso)" if w == math.inf else f"{w:g}"),
                           expandidos=b.n_expandidos, custo=b.custo))
    df = pd.DataFrame(linhas)
    df["razao_custo"] = df["custo"] / df["custo"].iloc[0]
    return df


def grafico_espectro(df, titulo="O espectro do A* ponderado"):
    """Barras de nós expandidos e linha da razão custo/ótimo, no mesmo eixo x."""
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    cores = [COR["Dijkstra"] if w == 0 else COR["A*"] if w == 1 else COR["Guloso"] if w == math.inf else COR["A* ponderado"]
             for w in df["w"]]
    fig.add_bar(x=df["rotulo"], y=df["expandidos"], marker_color=cores, name="nós expandidos",
                hovertemplate="w = %{x}<br>%{y} nós expandidos<extra></extra>")
    fig.add_scatter(x=df["rotulo"], y=df["razao_custo"], mode="lines+markers", name="custo / ótimo",
                    line=dict(color=COR["destino"], width=3), marker=dict(size=9),
                    hovertemplate="w = %{x}<br>custo = %{y:.3f} × ótimo<extra></extra>", secondary_y=True)
    fig.add_hline(y=1.0, line_dash="dot", line_color=COR["destino"], secondary_y=True, opacity=0.5)
    fig.update_layout(title=titulo, xaxis_title="peso w da heurística (f = g + w·h)", height=440,
                      legend=dict(orientation="h", y=1.08, x=0), **LAYOUT_CLARO)
    fig.update_yaxes(title_text="nós expandidos", secondary_y=False)
    teto = max(1.35, float(df["razao_custo"].max()) * 1.05, float(df.get("limite_teorico", df["razao_custo"]).max()) * 1.05)
    fig.update_yaxes(title_text="custo do caminho / ótimo", secondary_y=True, range=[0.95, teto])
    return fig

In [ ]:
df_w = espectro_w(G_tab, h_manhattan)
grafico_espectro(df_w).show()
df_w

## 3.3 · Laboratório

Agora é com você. Os controles abaixo rodam Dijkstra, a busca que você configurar e o guloso no mesmo tabuleiro. Quatro experimentos para começar.

1. **Dominância.** Com $w = 1$, troque a heurística entre Chebyshev, Euclidiana e Manhattan e observe os nós expandidos caírem sem o custo mudar.
2. **A inadmissível.** Escolha "Manhattan × 2" e procure uma semente em que o custo fique acima do ótimo. Depois volte para Manhattan com $w = 2$. É a mesma coisa? Por quê?
3. **Densidade.** Com Manhattan e $w = 1$, aumente a densidade de obstáculos. A vantagem do A\* sobre o Dijkstra cresce ou diminui? Pense no que os obstáculos fazem com $h^*$ enquanto $h$ fica igual.
4. **Oito direções.** Marque "8 direções" e compare Manhattan com Chebyshev. Uma delas deixou de ser admissível. Qual, e como você percebe no resultado?

In [ ]:
def laboratorio(heuristica="Manhattan", w=1.0, densidade=0.28, semente=13, diagonais=False):
    """Executa Dijkstra, a busca escolhida e o guloso no mesmo tabuleiro e desenha os três."""
    G = criar_tabuleiro(densidade=densidade, semente=semente, diagonais=diagonais)
    o, d = G.graph["origem"], G.graph["destino"]
    h = HEURISTICAS_GRADE[heuristica]
    buscas = [a_star(G, o, d),
              a_star(G, o, d, heuristica=h, w=w, nome=f"A* · {heuristica} · w={w:g}"),
              a_star(G, o, d, heuristica=h, w=math.inf)]
    cores = [COR["Dijkstra"], COR["A*"] if w == 1 else COR["A* ponderado"], COR["Guloso"]]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
    for ax, b, c in zip(axes, buscas, cores):
        desenhar_tabuleiro(G, b, ax=ax, cor=c)
    otimo = buscas[0].custo
    for ax, b in zip(axes, buscas):
        if b.custo > otimo:
            ax.set_title(ax.get_title() + f"  (+{100*(b.custo/otimo-1):.0f}% acima do ótimo)", fontsize=11, loc="left", color=COR["destino"])
    plt.tight_layout()
    plt.show()

widgets.interact(
    laboratorio,
    heuristica=widgets.Dropdown(options=list(HEURISTICAS_GRADE), value="Manhattan", description="heurística"),
    w=widgets.FloatSlider(min=0, max=5, step=0.25, value=1.0, description="peso w"),
    densidade=widgets.FloatSlider(min=0.0, max=0.45, step=0.05, value=0.28, description="obstáculos"),
    semente=widgets.IntSlider(min=0, max=50, value=13, description="semente"),
    diagonais=widgets.Checkbox(value=False, description="8 direções"),
);

# 4 · Natal, o mapa real

Do tabuleiro para a cidade. O **OSMnx** (Boeing, 2017; 2025) consulta o OpenStreetMap e devolve a malha viária como um `MultiDiGraph` do NetworkX. Dirigido porque há ruas de mão única. Multigrafo porque dois cruzamentos podem estar ligados por mais de uma via. Cada nó é um cruzamento (ou um fim de rua) com `x` (longitude) e `y` (latitude) em graus. Cada aresta é um trecho de via com `length` em metros, `highway` (a classe da via), `speed_kph` (velocidade informada no OSM ou imputada pela classe) e `travel_time` em segundos.

Uma decisão prática antes de qualquer coisa. Baixar a malha de uma cidade leva um ou dois minutos e depende de um servidor público (a Overpass API) que não gosta de trinta pedidos simultâneos vindos da mesma sala de aula. Por isso a função abaixo procura primeiro uma cópia local, depois uma cópia no repositório da disciplina, e só então baixa. Depois de baixar, salva em GraphML. Guarde esse arquivo.

In [ ]:
LUGAR = "Natal, Rio Grande do Norte, Brazil"
ARQUIVO = "natal_drive.graphml"
URL_REPO = "https://raw.githubusercontent.com/ivanovitchm/datastructure/main/lessons/week09/natal_drive.graphml"

def carregar_grafo(lugar=LUGAR, arquivo=ARQUIVO, url=URL_REPO):
    """Carrega o grafo viário tentando, nesta ordem, o arquivo local, a cópia no repositório
    da disciplina e, por fim, o download do OpenStreetMap (Overpass API).

    Guardar o GraphML evita que uma turma inteira dependa do servidor do OSM ao mesmo tempo
    e garante que todos trabalhem exatamente com o mesmo grafo.
    """
    import os, urllib.request
    if os.path.exists(arquivo):
        print(f"Lendo {arquivo} (cópia local).")
        return ox.load_graphml(arquivo)
    if url:
        try:
            urllib.request.urlretrieve(url, arquivo)
            print(f"Lendo {arquivo} (cópia do repositório da disciplina).")
            return ox.load_graphml(arquivo)
        except Exception:
            print("Sem cópia no repositório. Baixando do OpenStreetMap, isso leva um ou dois minutos ...")
    G = ox.graph_from_place(lugar, network_type="drive")
    G = ox.routing.add_edge_speeds(G)          # imputa velocidade (km/h) onde o OSM não informa
    G = ox.routing.add_edge_travel_times(G)    # travel_time (s) = length / speed
    ox.save_graphml(G, arquivo)
    print(f"Grafo salvo em {arquivo}. Vale a pena guardar este arquivo no repositório.")
    return G

G = carregar_grafo()
print(f"{type(G).__name__} com {G.number_of_nodes():,} nós e {G.number_of_edges():,} arestas")

## 4.1 · O que há dentro do grafo

Um nó, uma aresta e a distribuição de velocidades. A velocidade máxima, `V_MAX_KPH`, vai ser importante na seção 5. Repare também no `street_count` do nó (quantas vias chegam nele, contando as que o OSMnx simplificou) e no fato de `highway` poder ser uma lista quando o trecho foi consolidado a partir de segmentos de classes diferentes.

In [ ]:
# O que há em um nó e em uma aresta. Os nós têm x (longitude) e y (latitude) em graus.
no_exemplo = next(iter(G.nodes))
print("nó", no_exemplo, G.nodes[no_exemplo])
u, v, k = next(iter(G.edges(keys=True)))
print("aresta", (u, v, k), {ch: val for ch, val in G.edges[u, v, k].items() if ch != "geometry"})

velocidades = pd.Series(nx.get_edge_attributes(G, "speed_kph"))
print("\nVelocidades (km/h) nas arestas:")
print(velocidades.describe().round(1).to_string())
V_MAX_KPH = float(velocidades.max())

## 4.2 · Pontos de referência

Dez lugares de Natal, em coordenadas aproximadas de propósito. Um algoritmo de rotas não trabalha com coordenadas, trabalha com nós, e o primeiro passo de qualquer aplicativo de navegação é **encaixar** (snap) a coordenada no nó mais próximo da malha. `ox.distance.nearest_nodes` faz isso e, com `return_dist=True`, informa a distância do encaixe. Se um ponto ficou a 400 m do nó mais próximo, a rota vai começar 400 m longe do lugar real. Olhe essa coluna. Em um aplicativo, ela vira o "caminhe até a rua tal" do início do trajeto.

In [ ]:
# Pontos de referência de Natal (latitude, longitude). As coordenadas são aproximadas de propósito.
# O passo seguinte "encaixa" cada ponto no nó da malha viária mais próximo e informa a distância
# do encaixe, para você saber quão longe do lugar real a rota começa ou termina.
PONTOS = {
    "UFRN (Campus Central)":          (-5.8393, -35.1996),
    "Ponta Negra (Morro do Careca)":  (-5.8805, -35.1680),
    "Forte dos Reis Magos":           (-5.7578, -35.1947),
    "Arena das Dunas":                (-5.8289, -35.2125),
    "Midway Mall":                    (-5.8130, -35.2055),
    "Praia da Redinha":               (-5.7555, -35.2045),
    "Rodoviária":                     (-5.8200, -35.2335),
    "Cidade Alta (Centro Histórico)": (-5.7840, -35.2070),
    "Ponto A da aula (Capim Macio)":  (-5.8428893, -35.1998821),
    "Ponto B da aula (Zona Norte)":   (-5.719613, -35.268811),
}

def encaixar_pontos(G, pontos):
    """Para cada ponto (lat, lon) devolve o nó mais próximo e a distância do encaixe em metros."""
    nomes = list(pontos)
    lats = [pontos[n][0] for n in nomes]
    lons = [pontos[n][1] for n in nomes]
    nos, dists = ox.distance.nearest_nodes(G, lons, lats, return_dist=True)
    return {n: no for n, no in zip(nomes, nos)}, {n: d for n, d in zip(nomes, dists)}

NOS, ENCAIXE = encaixar_pontos(G, PONTOS)
pd.DataFrame({"nó": NOS, "distância do encaixe (m)": pd.Series(ENCAIXE).round(0)})

In [ ]:
xy = {n: (d["y"], d["x"]) for n, d in G.nodes(data=True)}   # nó -> (lat, lon)
fig, ax = plt.subplots(figsize=(8, 9), facecolor=COR["fundo_mapa"])
ax.set_facecolor(COR["fundo_mapa"])
ox.plot_graph(G, ax=ax, node_size=0, edge_color=COR["ruas"], edge_linewidth=0.35,
              bgcolor=COR["fundo_mapa"], show=False, close=False)
for nome, no in NOS.items():
    ax.scatter(xy[no][1], xy[no][0], s=70, c=COR["fronteira"], edgecolors="white", linewidths=1, zorder=5)
    ax.annotate(nome.split(" (")[0], (xy[no][1], xy[no][0]), xytext=(7, 4), textcoords="offset points",
                color="white", fontsize=9)
ax.set_title(f"Malha viária de Natal · {G.number_of_nodes():,} nós · {G.number_of_edges():,} arestas",
             color="white", loc="left")
plt.show()

# 5 · O experimento que dá errado de propósito

Os slides desta semana trazem uma tabela de desempenho com quatro heurísticas e dois pesos sobre este mesmo mapa. Ela tem dois números estranhos, e quem sabe lê-los aprende mais sobre o A\* do que em qualquer prova de teorema.

Primeiro, **euclidiana, Manhattan e Dijkstra produzem exatamente a mesma rota, com o mesmo comprimento e o mesmo tempo**, para os dois pesos. Segundo, **great circle com `travel_time` roda dez vezes mais rápido que todo mundo e entrega uma rota 25% mais lenta que a do Dijkstra**. Antes de ler a explicação, pare um minuto e tente formulá-la. A pista está na pergunta que dá nome à seção 5.1.

Vamos usar o mesmo par origem-destino da aula, o ponto A em Capim Macio e o ponto B na Zona Norte.

In [ ]:
ORIGEM = NOS["Ponto A da aula (Capim Macio)"]
DESTINO = NOS["Ponto B da aula (Zona Norte)"]
reta_m = ox.distance.great_circle(G.nodes[ORIGEM]["y"], G.nodes[ORIGEM]["x"],
                                  G.nodes[DESTINO]["y"], G.nodes[DESTINO]["x"])
print(f"Distância em linha reta entre origem e destino: {reta_m/1000:.1f} km")

## 5.1 · Em que unidade está a heurística?

O peso `length` está em **metros**. O peso `travel_time` está em **segundos**. E as heurísticas?

* `ox.distance.euclidean(lat1, lon1, lat2, lon2)` calcula $\sqrt{\Delta\text{lat}^2 + \Delta\text{lon}^2}$ sobre o que você passar. Passamos graus, ela devolve **graus**. Entre os dois pontos da aula dá algo como $0{,}14$. Somar $0{,}14$ a um $g$ de $15\,000$ metros não muda a ordem de ninguém na fila. A heurística é admissível, consistente e **inútil**. O A\* vira Dijkstra.
* A Manhattan em `x` e `y` tem o mesmo problema. Graus.
* `ox.distance.great_circle` devolve **metros**. Contra `length` é a heurística certa. Contra `travel_time` compara metros com segundos. A 50 km/h, um metro equivale a $0{,}072$ segundos, logo $h$ está superestimando por um fator de cerca de 14. É uma heurística violentamente inadmissível, e o A\* se comporta como o guloso. Rápido e errado.

A tabela abaixo mostra $h(\text{origem}, \text{destino})$ para as três heurísticas originais, dividido pelo custo ótimo em cada peso. Uma heurística útil tem essa razão perto de 1 e abaixo de 1. Veja o que temos.

In [ ]:
def coordenadas(G):
    """Dicionário nó -> (lat, lon). Evita o acesso G.nodes[u]['y'] a cada chamada da heurística."""
    return {n: (d["y"], d["x"]) for n, d in G.nodes(data=True)}


def h_euclidiana_graus(G):
    """A heurística do notebook original. Euclidiana em (lat, lon), portanto em GRAUS."""
    xy = coordenadas(G)
    return lambda u, v: ox.distance.euclidean(xy[u][0], xy[u][1], xy[v][0], xy[v][1])


def h_manhattan_graus(G):
    """Manhattan em (lat, lon), também em graus."""
    xy = coordenadas(G)
    return lambda u, v: abs(xy[u][1] - xy[v][1]) + abs(xy[u][0] - xy[v][0])


def h_grande_circulo(G):
    """Distância ortodrômica (great circle) em METROS entre dois nós."""
    xy = coordenadas(G)
    return lambda u, v: ox.distance.great_circle(xy[u][0], xy[u][1], xy[v][0], xy[v][1])


def h_euclidiana_metros(G_proj):
    """Euclidiana sobre um grafo PROJETADO (x, y em metros)."""
    xy = {n: (d["x"], d["y"]) for n, d in G_proj.nodes(data=True)}
    return lambda u, v: math.hypot(xy[u][0] - xy[v][0], xy[u][1] - xy[v][1])


def h_manhattan_metros(G_proj):
    """Manhattan sobre um grafo projetado. Em metros, mas NÃO admissível para ruas diagonais."""
    xy = {n: (d["x"], d["y"]) for n, d in G_proj.nodes(data=True)}
    return lambda u, v: abs(xy[u][0] - xy[v][0]) + abs(xy[u][1] - xy[v][1])


def h_tempo(G, v_max_kph=None):
    """Heurística em SEGUNDOS para o peso travel_time. Distância em linha reta dividida pela
    maior velocidade do grafo. Nenhum trajeto pode ser mais rápido que isso, logo é admissível."""
    if v_max_kph is None:
        v_max_kph = max(d["speed_kph"] for _, _, d in G.edges(data=True))
    v_max = v_max_kph / 3.6
    dist = h_grande_circulo(G)
    return lambda u, v: dist(u, v) / v_max

In [ ]:
def resumo_rota(G, rota):
    """Comprimento total (m) e tempo total (s) de uma rota, somando os atributos das arestas.
    Em multigrafos escolhe, entre arestas paralelas, a de menor comprimento."""
    comprimento = tempo = 0.0
    for u, v in zip(rota[:-1], rota[1:]):
        dados = G.adj[u][v]
        melhor = min(dados.values(), key=lambda d: d["length"]) if G.is_multigraph() else dados
        comprimento += melhor["length"]
        tempo += melhor["travel_time"]
    return comprimento, tempo

In [ ]:
# As heurísticas exatamente como estavam no notebook original, lado a lado com o custo ótimo.
otimo_len = nx.dijkstra_path_length(G, ORIGEM, DESTINO, weight="length")
otimo_tempo = nx.dijkstra_path_length(G, ORIGEM, DESTINO, weight="travel_time")

h_gc = h_grande_circulo(G)      # usada em todo o resto do notebook
heuristicas_originais = {
    "euclidiana (lat, lon)": (h_euclidiana_graus(G), "graus"),
    "Manhattan (lat, lon)":  (h_manhattan_graus(G), "graus"),
    "great circle":          (h_gc, "metros"),
}
linhas = []
for nome, (h, unidade) in heuristicas_originais.items():
    valor = h(ORIGEM, DESTINO)
    linhas.append({"heurística": nome, "unidade": unidade, "h(origem, destino)": valor,
                   "h / h* com peso length (m)": valor / otimo_len,
                   "h / h* com peso travel_time (s)": valor / otimo_tempo})
tabela_h = pd.DataFrame(linhas)
print(f"custo ótimo em metros: {otimo_len:,.0f} m | custo ótimo em segundos: {otimo_tempo:,.0f} s")
tabela_h.style.format({"h(origem, destino)": "{:,.4f}", "h / h* com peso length (m)": "{:.4f}",
                       "h / h* com peso travel_time (s)": "{:.3f}"}).background_gradient(
    subset=["h / h* com peso length (m)", "h / h* com peso travel_time (s)"], cmap="Reds", vmin=0, vmax=2)

## 5.2 · A tabela dos slides, com a coluna que faltava

Reproduzimos a tabela com as heurísticas originais, agora medindo também os **nós expandidos** (que a função `nx.astar_path` não expõe, mas a nossa implementação sim). O tempo de parede é medido na implementação do NetworkX, repetido três vezes, porque é o tempo que você veria na prática.

Leitura esperada. Euclidiana e Manhattan em graus expandem **exatamente** o mesmo número de nós que o Dijkstra. Great circle com `length` expande bem menos e mantém o ótimo. Great circle com `travel_time` expande pouquíssimo e perde o ótimo (a coluna `otimo` fica em vermelho e `custo_vs_otimo` sobe).

In [ ]:
def avaliar(G, origem, destino, heuristica, peso, nome, w=1.0, repeticoes=3):
    """Mede uma configuração (heurística, peso, w) em um par origem-destino.

    Nós expandidos, custo e caminho vêm da nossa implementação instrumentada. O tempo de
    parede vem da implementação de referência do NetworkX (`nx.astar_path`, ou
    `nx.dijkstra_path` sem heurística), repetido `repeticoes` vezes, porque é o tempo que
    você veria "na prática". Para w diferente de 1 (A* ponderado, guloso), que o NetworkX
    não oferece, o tempo é o da nossa implementação.
    """
    b = a_star(G, origem, destino, heuristica=heuristica, peso=peso, w=w, nome=nome)
    if heuristica is None or w == 0:
        rodar = lambda: nx.dijkstra_path(G, origem, destino, weight=peso)
    elif w == 1:
        rodar = lambda: nx.astar_path(G, origem, destino, heuristic=heuristica, weight=peso)
    else:
        rodar = lambda: a_star(G, origem, destino, heuristica=heuristica, peso=peso, w=w)
    tempos = timeit.repeat(rodar, number=1, repeat=repeticoes)
    comprimento, tempo_rota = resumo_rota(G, b.caminho)
    return dict(heuristica=nome, peso=peso, tempo_ms=1000 * np.mean(tempos), desvio_ms=1000 * np.std(tempos),
                nos_expandidos=b.n_expandidos, custo=b.custo, distancia_m=comprimento,
                tempo_rota_s=tempo_rota, _busca=b)


def tabela_avaliacao(resultados):
    """DataFrame limpo a partir da lista de dicionários de `avaliar`, com as colunas
    'otimo' (achou o menor custo daquele peso?) e 'custo_vs_otimo'."""
    df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in resultados])
    otimos = df.groupby("peso")["custo"].transform("min")
    df["otimo"] = np.isclose(df["custo"], otimos, rtol=1e-9)
    df["custo_vs_otimo"] = df["custo"] / otimos
    return df

In [ ]:
# Reproduzindo a tabela dos slides, agora com a coluna que faltava: nós expandidos.
configuracoes = []
for peso in ("length", "travel_time"):
    for nome, (h, _) in heuristicas_originais.items():
        configuracoes.append((nome, h, peso))
    configuracoes.append(("Dijkstra", None, peso))

resultados_originais = [avaliar(G, ORIGEM, DESTINO, h, peso, nome) for nome, h, peso in configuracoes]
tabela_original = tabela_avaliacao(resultados_originais)
estilizar(tabela_original.drop(columns=["custo"]))

## 5.3 · Corrigindo

Duas correções, uma para cada peso.

**Para `length`.** Ou usamos `great_circle` (já em metros), ou **projetamos o grafo** com `ox.project_graph`, que converte latitude e longitude para um sistema de coordenadas UTM em metros. Com o grafo projetado, a euclidiana em `x` e `y` passa a estar em metros e vira uma heurística admissível e barata (sem trigonometria). A Manhattan em metros também fica na escala certa, mas repare no resultado dela. Ruas não seguem uma grade perfeita, e a distância de Manhattan **pode superestimar** o comprimento de um trajeto diagonal. Na cidade ela não é admissível, e a tabela vai mostrar isso.

**Para `travel_time`.** A estimativa de tempo tem que ser em segundos e não pode superestimar. Nenhum trajeto pode ser mais rápido do que percorrer a linha reta na **maior velocidade que existe no grafo**. Logo

$$h_{\text{tempo}}(n) = \frac{\text{great\_circle}(n, \text{destino})}{v_{\max}}$$

é admissível e consistente. Mas repare no preço. Se $v_{\max}$ é 80 km/h e a maior parte da cidade anda a 40, a heurística subestima por um fator de 2 em quase todo lugar. Admissível, porém **fraca**. Veremos na tabela que ela expande bem mais nós do que a de distância, e na seção 5.6 vamos negociar isso.

In [ ]:
# Correção 1: projetar o grafo para um sistema métrico (UTM). Agora x e y estão em metros.
G_proj = ox.project_graph(G)
print("CRS do grafo projetado:", G_proj.graph["crs"])

# Correção 2: heurísticas coerentes com a unidade do peso. Cada entrada diz também em qual
# grafo a busca deve rodar (o projetado, para as heurísticas que leem x e y em metros).
heuristicas_corrigidas = [
    # (nome,                        heurística,                     peso,          grafo)
    ("euclidiana (m)",              h_euclidiana_metros(G_proj),    "length",      G_proj),
    ("Manhattan (m)",               h_manhattan_metros(G_proj),     "length",      G_proj),
    ("great circle (m)",            h_grande_circulo(G),            "length",      G),
    ("Dijkstra",                    None,                           "length",      G),
    ("great circle / v_max (s)",    h_tempo(G, V_MAX_KPH),          "travel_time", G),
    ("Dijkstra",                    None,                           "travel_time", G),
]
resultados_corrigidos = [avaliar(grafo, ORIGEM, DESTINO, h, peso, nome)
                         for nome, h, peso, grafo in heuristicas_corrigidas]
tabela_corrigida = tabela_avaliacao(resultados_corrigidos)
estilizar(tabela_corrigida.drop(columns=["custo"]))

## 5.4 · A pegada de busca

Agora a figura que resume a aula. Cada ponto é um nó expandido, pintado do tom claro ao tom cheio na ordem de expansão. A rota final é a linha grossa. Fundo escuro, figura clara.

E a promessa do fechamento. **Por que a pegada do A\* é uma elipse?** O A\* expande os nós com $f(n) = g(n) + h(n) < C^*$. Se $h$ é a distância em linha reta e, numa aproximação grosseira, $g(n)$ também cresce como a distância em linha reta desde a origem, então a região expandida é

$$\{\, n : d(\text{origem}, n) + d(n, \text{destino}) \le C^* \,\},$$

que é exatamente a definição de uma **elipse com focos na origem e no destino** e eixo maior $C^*$. Quanto mais direto o caminho ótimo (quanto menor $C^*$ em relação à distância em linha reta), mais fina a elipse. O Dijkstra, com $h = 0$, expande $\{ n : d(\text{origem}, n) \le C^* \}$, um **disco** de raio $C^*$ centrado na origem. A elipse cabe dentro do disco. Sempre.

In [ ]:
def desenhar_pegada(G, buscas, cores=None, figsize=None, titulo=None, tamanho_ponto=3):
    """Pegada de busca sobre o mapa. Cada nó expandido é um ponto, pintado pela ordem de expansão
    (claro no início, cheio no fim). A rota final é a linha grossa. Fundo escuro, figura clara."""
    nomes = list(buscas)
    cores = cores or [COR.get(n.split(" ")[0], COR["A*"]) for n in nomes]
    xy = coordenadas(G)
    figsize = figsize or (6.2 * len(nomes), 6.8)
    fig, axes = plt.subplots(1, len(nomes), figsize=figsize, facecolor=COR["fundo_mapa"])
    axes = np.atleast_1d(axes)
    for ax, nome, cor in zip(axes, nomes, cores):
        b = buscas[nome]
        ax.set_facecolor(COR["fundo_mapa"])
        ox.plot_graph(G, ax=ax, node_size=0, edge_color=COR["ruas"], edge_linewidth=0.35,
                      bgcolor=COR["fundo_mapa"], show=False, close=False)
        xs = [xy[n][1] for n in b.expandidos]
        ys = [xy[n][0] for n in b.expandidos]
        ax.scatter(xs, ys, c=np.arange(len(xs)), cmap=mapa_de_cores(cor), s=tamanho_ponto, linewidths=0, zorder=3)
        if b.caminho:
            ax.plot([xy[n][1] for n in b.caminho], [xy[n][0] for n in b.caminho], color="white", lw=4.5, zorder=4, alpha=0.9)
            ax.plot([xy[n][1] for n in b.caminho], [xy[n][0] for n in b.caminho], color=cor, lw=2.5, zorder=5)
            o, d = b.caminho[0], b.caminho[-1]
            ax.scatter([xy[o][1]], [xy[o][0]], s=90, c="white", edgecolors=COR["origem"], linewidths=1.5, zorder=6)
            ax.scatter([xy[d][1]], [xy[d][0]], s=90, c=COR["destino"], edgecolors="white", linewidths=1.5, zorder=6)
        ax.set_title(f"{nome}\n{b.n_expandidos:,} nós expandidos · custo {b.custo:,.0f}",
                     color="white", fontsize=12, loc="left")
    if titulo:
        fig.suptitle(titulo, color="white", fontsize=14, x=0.01, y=0.99, ha="left")
        plt.tight_layout(rect=(0, 0, 1, 0.94))
    else:
        plt.tight_layout()
    return fig, axes

In [ ]:
buscas_natal = {
    "Dijkstra": a_star(G, ORIGEM, DESTINO, peso="length"),
    "A*":       a_star(G, ORIGEM, DESTINO, heuristica=h_gc, peso="length"),
    "Guloso":   a_star(G, ORIGEM, DESTINO, heuristica=h_gc, peso="length", w=math.inf),
}
fig, _ = desenhar_pegada(G, buscas_natal, tamanho_ponto=4,
                         titulo="Pegada de busca sobre Natal · peso = length · heurística = great circle")
plt.show()

## 5.5 · A fronteira crescendo

A mesma pegada, agora no tempo. Aperte play e veja a frente de onda do A\* se alongar na direção do destino, e depois compare com a do Dijkstra se espalhando em todas as direções. As duas animações usam uma amostra dos nós expandidos para manter a figura leve.

In [ ]:
def animar_fronteira_mapa(G, busca, cor=None, quadros=36, max_pontos=12000, altura=640, titulo=None):
    """Animação da fronteira de busca crescendo sobre o mapa (Plotly, WebGL).

    Para manter a figura leve, a lista de nós expandidos é subamostrada até `max_pontos`.
    A rota aparece no último quadro.
    """
    cor = cor or COR.get(busca.nome.split(" ")[0], COR["A*"])
    xy = coordenadas(G)
    # fundo: todas as arestas como uma única linha com separadores None
    ex, ey = [], []
    for u, v in G.edges():
        ex += [xy[u][1], xy[v][1], None]
        ey += [xy[u][0], xy[v][0], None]
    exp = busca.expandidos
    if len(exp) > max_pontos:
        exp = exp[:: math.ceil(len(exp) / max_pontos)]
    n = len(exp)
    cortes = np.unique(np.linspace(0, n, quadros + 1).astype(int))
    xs = np.array([xy[k][1] for k in exp]); ys = np.array([xy[k][0] for k in exp])

    def pontos(i):
        return go.Scattergl(x=xs[:i], y=ys[:i], mode="markers",
                            marker=dict(size=4, color=np.arange(i), colorscale=[[0, tonalidade(cor, 0.2)], [1, cor]],
                                        cmin=0, cmax=max(n, 1)),
                            hoverinfo="skip", name="expandidos")

    rota_x = [xy[k][1] for k in busca.caminho]; rota_y = [xy[k][0] for k in busca.caminho]
    fig = go.Figure(data=[
        go.Scattergl(x=ex, y=ey, mode="lines", line=dict(color=COR["ruas"], width=0.6), hoverinfo="skip", name="ruas"),
        pontos(0),
        go.Scattergl(x=[], y=[], mode="lines", line=dict(color="white", width=4), hoverinfo="skip", name="rota"),
        go.Scattergl(x=[rota_x[0], rota_x[-1]], y=[rota_y[0], rota_y[-1]], mode="markers",
                     marker=dict(size=[13, 13], color=["white", COR["destino"]], line=dict(color=[COR["origem"], "white"], width=2)),
                     hoverinfo="skip", name="origem/destino"),
    ])
    frames = [go.Frame(name=str(c), traces=[1, 2],
                       data=[pontos(c), go.Scattergl(x=rota_x if c == n else [], y=rota_y if c == n else [])])
              for c in cortes]
    fig.frames = frames
    fig.update_layout(
        title=titulo or f"{busca.nome} · {busca.n_expandidos:,} nós expandidos · custo {busca.custo:,.0f}",
        height=altura, paper_bgcolor=COR["fundo_mapa"], plot_bgcolor=COR["fundo_mapa"], font=dict(color="white"),
        showlegend=False, margin=dict(l=10, r=10, t=60, b=10),
        xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
        updatemenus=[dict(type="buttons", showactive=False, x=0, y=0, xanchor="left", yanchor="bottom",
                          bgcolor="#1f2430", font=dict(color="white"),
                          buttons=[dict(label="▶ play", method="animate",
                                        args=[None, dict(frame=dict(duration=90, redraw=True), fromcurrent=True,
                                                         transition=dict(duration=0))]),
                                   dict(label="❚❚ pausa", method="animate",
                                        args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
        sliders=[dict(active=0, x=0.18, len=0.8, y=0.0, yanchor="bottom", font=dict(color="white"),
                      currentvalue=dict(prefix="nós expandidos (amostra): ", visible=True),
                      steps=[dict(method="animate", label=str(c),
                                  args=[[str(c)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                                      transition=dict(duration=0))]) for c in cortes])],
    )
    return fig

In [ ]:
animar_fronteira_mapa(G, buscas_natal["A*"]).show()

In [ ]:
animar_fronteira_mapa(G, buscas_natal["Dijkstra"]).show()

## 5.6 · Qual velocidade assumir?

A heurística de tempo admissível usa $v_{\max}$ e por isso é fraca. O que acontece se assumirmos uma velocidade menor $v$ na fórmula? Como $h_v = \frac{d}{v} = \frac{v_{\max}}{v} \cdot h_{v_{\max}}$, isso é **exatamente o A\* ponderado com $w = v_{\max} / v$**. A garantia de subotimalidade limitada se traduz em linguagem de trânsito. Se a heurística assume 40 km/h numa cidade cujo limite máximo é 80, o caminho devolvido demora **no máximo duas vezes** o ótimo. Na prática, bem menos. O gráfico abaixo mostra os nós expandidos (barras), o custo real dividido pelo ótimo (linha cheia) e o limite teórico $v_{\max}/v$ (linha tracejada).

In [ ]:
# Para travel_time, a única heurística admissível assume a MAIOR velocidade do grafo em todo lugar.
# O que acontece se assumirmos uma velocidade menor? A heurística fica mais "apertada", porém inadmissível.
velocidades_teste = sorted({V_MAX_KPH} | {v for v in (90, 70, 60, 50, 40, 30) if v < V_MAX_KPH}, reverse=True)
linhas = []
for v in velocidades_teste:
    b = a_star(G, ORIGEM, DESTINO, heuristica=h_tempo(G, v), peso="travel_time", nome=f"v={v:g}")
    linhas.append(dict(w=V_MAX_KPH / v, rotulo=f"{v:g} km/h", expandidos=b.n_expandidos, custo=b.custo))
df_vel = pd.DataFrame(linhas)
df_vel["razao_custo"] = df_vel["custo"] / otimo_tempo
df_vel["limite_teorico"] = df_vel["w"]
fig = grafico_espectro(df_vel, titulo="Que velocidade assumir na heurística de tempo?")
fig.add_scatter(x=df_vel["rotulo"], y=df_vel["limite_teorico"], mode="lines", name="limite teórico v_max / v",
                line=dict(color=COR["destino"], dash="dash", width=1.5), secondary_y=True)
fig.update_layout(xaxis_title="velocidade assumida na heurística (h = distância / v)")
fig.show()
df_vel.round(3)

## 5.7 · Mais curta ou mais rápida?

O caminho mais curto em metros e o mais rápido em segundos raramente coincidem. Um pega as avenidas, o outro corta por dentro. Sobre o mapa real (tiles CARTO), com a legenda informando quilômetros e minutos de cada um. Use o zoom.

In [ ]:
def rota_sobre_mapa(G, rotas, nomes=None, cores=None, zoom=11.6, altura=620):
    """Rotas sobre um mapa real (tiles CARTO dark, via MapLibre). Precisa de Internet no navegador."""
    xy = coordenadas(G)
    nomes = nomes or [f"rota {i+1}" for i in range(len(rotas))]
    cores = cores or [COR["A*"], COR["Dijkstra"], COR["Guloso"], COR["A* ponderado"]]
    fig = go.Figure()
    for rota, nome, cor in zip(rotas, nomes, cores):
        comprimento, tempo = resumo_rota(G, rota)
        fig.add_trace(go.Scattermap(lat=[xy[n][0] for n in rota], lon=[xy[n][1] for n in rota], mode="lines",
                                    line=dict(width=5, color=cor),
                                    name=f"{nome} · {comprimento/1000:.1f} km · {tempo/60:.0f} min"))
    o, d = rotas[0][0], rotas[0][-1]
    fig.add_trace(go.Scattermap(lat=[xy[o][0], xy[d][0]], lon=[xy[o][1], xy[d][1]], mode="markers+text",
                                marker=dict(size=14, color=["white", COR["destino"]]),
                                text=["origem", "destino"], textposition="top right",
                                textfont=dict(color="white"), name="origem e destino"))
    lats = [xy[n][0] for r in rotas for n in r]; lons = [xy[n][1] for r in rotas for n in r]
    fig.update_layout(map=dict(style="carto-darkmatter", zoom=zoom,
                               center=dict(lat=float(np.mean(lats)), lon=float(np.mean(lons)))),
                      height=altura, margin=dict(l=0, r=0, t=0, b=0),
                      legend=dict(x=0.01, y=0.99, bgcolor="rgba(15,17,23,0.75)", font=dict(color="white")))
    return fig

In [ ]:
rota_curta = buscas_natal["A*"].caminho
rota_rapida = a_star(G, ORIGEM, DESTINO, heuristica=h_tempo(G, V_MAX_KPH), peso="travel_time").caminho
rota_sobre_mapa(G, [rota_curta, rota_rapida], nomes=["mais curta (length)", "mais rápida (travel_time)"],
                cores=[COR["A*"], COR["A* ponderado"]]).show()

## 5.8 · Explore

Escolha origem, destino, peso e o $w$ da heurística. O painel da esquerda é sempre o Dijkstra, para referência. Tente um par próximo (UFRN e Arena das Dunas) e um par distante (Ponta Negra e Praia da Redinha), e veja como a razão entre as duas pegadas muda. Tente também `travel_time` com $w = 1$ e depois $w = 2$.

In [ ]:
def explorar_rota(origem="UFRN (Campus Central)", destino="Forte dos Reis Magos", peso="length", w=1.0):
    """Dijkstra e A* (com o peso w escolhido) entre dois pontos de Natal, lado a lado."""
    o, d = NOS[origem], NOS[destino]
    if o == d:
        print("Escolha dois pontos diferentes."); return
    h = h_gc if peso == "length" else h_tempo(G, V_MAX_KPH)
    nome = "A*" if w == 1 else ("Guloso" if w == math.inf else f"A* ponderado w={w:g}")
    buscas = {"Dijkstra": a_star(G, o, d, peso=peso), nome: a_star(G, o, d, heuristica=h, peso=peso, w=w)}
    cores = [COR["Dijkstra"], COR["A*"] if w == 1 else COR["Guloso"] if w == math.inf else COR["A* ponderado"]]
    desenhar_pegada(G, buscas, cores=cores, tamanho_ponto=4,
                    titulo=f"{origem} → {destino} · peso = {peso}")
    plt.show()
    for n, b in buscas.items():
        dist_m, tempo_s = resumo_rota(G, b.caminho)
        print(f"{n:22s} {b.n_expandidos:6,} nós expandidos | {dist_m/1000:5.1f} km | {tempo_s/60:5.1f} min | {b.tempo_s*1000:6.0f} ms")

widgets.interact(
    explorar_rota,
    origem=widgets.Dropdown(options=list(PONTOS), value="UFRN (Campus Central)", description="origem"),
    destino=widgets.Dropdown(options=list(PONTOS), value="Forte dos Reis Magos", description="destino"),
    peso=widgets.ToggleButtons(options=["length", "travel_time"], description="peso"),
    w=widgets.SelectionSlider(options=[0.5, 1.0, 1.5, 2.0, 3.0, math.inf], value=1.0, description="peso w"),
);

# 6 · Bancada de desempenho

Um par origem-destino é uma anedota. Doze pares são um experimento. A bancada abaixo roda quatro configurações em doze pares sorteados entre os pontos de referência e registra, para cada combinação, o tempo de parede, os nós expandidos, o custo e a razão para o ótimo.

Duas coisas para ter em mente ao ler.

**Tempo de parede engana.** O número de nós expandidos mede o algoritmo. O tempo mede o algoritmo **mais** a implementação, a linguagem e o custo de cada chamada da heurística. Em Python, uma chamada de `great_circle` envolve seis funções trigonométricas e custa mais que uma iteração inteira do Dijkstra. Um A\* que expande metade dos nós pode demorar quase o mesmo. A euclidiana sobre o grafo projetado é a mesma geometria sem trigonometria, e a tabela vai mostrar a diferença. Em C++ ou Rust, com coordenadas pré-carregadas em vetores, a conta muda completamente.

**Complexidade.** Com um heap binário, Dijkstra e A\* custam $O((V + E) \log V)$ no pior caso. A diferença entre eles não está na ordem de complexidade e sim na **constante**, que para o A\* é o tamanho do conjunto $\{n : f(n) < C^*\}$. Em uma cidade, com uma boa heurística, esse conjunto costuma ser uma fração pequena de $V$, e cresce bem mais devagar com a distância do que o disco do Dijkstra.

In [ ]:
def bancada(pares, configuracoes, repeticoes=2):
    """Roda todas as configurações em todos os pares e devolve um DataFrame longo.

    `pares` é uma lista de (nome_origem, nome_destino) e `configuracoes` uma lista de
    tuplas (nome, heurística, peso, grafo, w). A coluna custo_vs_otimo usa o Dijkstra
    do mesmo par e do mesmo peso como referência.
    """
    linhas = []
    for no_o, no_d in pares:
        o, d = NOS[no_o], NOS[no_d]
        for nome, h, peso, grafo, w in configuracoes:
            r = avaliar(grafo, o, d, h, peso, nome, w=w, repeticoes=repeticoes)
            r.pop("_busca")
            r.update(origem=no_o, destino=no_d, reta_km=h_gc(o, d) / 1000)
            linhas.append(r)
    df = pd.DataFrame(linhas)
    otimos = df.groupby(["origem", "destino", "peso"])["custo"].transform("min")
    df["custo_vs_otimo"] = df["custo"] / otimos
    return df

rng = random.Random(9)
nomes_pontos = list(PONTOS)
pares = rng.sample([(a, b) for a in nomes_pontos for b in nomes_pontos if a < b], 12)
configs = [
    ("Dijkstra",            None,                        "length", G,      1.0),
    ("A* great circle",     h_gc,                        "length", G,      1.0),
    ("A* euclidiana (m)",   h_euclidiana_metros(G_proj), "length", G_proj, 1.0),
    ("Guloso great circle", h_gc,                        "length", G,      math.inf),
]
df_bancada = bancada(pares, configs)
df_bancada.groupby("heuristica")[["tempo_ms", "nos_expandidos", "custo_vs_otimo"]].agg(["mean", "max"]).round(3)

In [ ]:
cores_alg = {"Dijkstra": COR["Dijkstra"], "A* great circle": COR["A*"], "A* euclidiana (m)": COR["A* ponderado"],
             "Guloso great circle": COR["Guloso"]}
fig = make_subplots(rows=1, cols=2, subplot_titles=["Nós expandidos × distância em linha reta",
                                                    "Tempo de parede (NetworkX) × distância em linha reta"])
for alg, grupo in df_bancada.groupby("heuristica"):
    grupo = grupo.sort_values("reta_km")
    hover = [f"{o} → {d}<br>{n:,} nós · {t:.0f} ms · custo/ótimo {c:.3f}"
             for o, d, n, t, c in zip(grupo.origem, grupo.destino, grupo.nos_expandidos, grupo.tempo_ms, grupo.custo_vs_otimo)]
    fig.add_scatter(x=grupo.reta_km, y=grupo.nos_expandidos, mode="markers", name=alg, legendgroup=alg,
                    marker=dict(size=10, color=cores_alg[alg]), text=hover, hovertemplate="%{text}<extra></extra>", row=1, col=1)
    fig.add_scatter(x=grupo.reta_km, y=grupo.tempo_ms, mode="markers", name=alg, legendgroup=alg, showlegend=False,
                    marker=dict(size=10, color=cores_alg[alg]), text=hover, hovertemplate="%{text}<extra></extra>", row=1, col=2)
fig.update_xaxes(title_text="distância em linha reta (km)")
fig.update_yaxes(title_text="nós expandidos", row=1, col=1)
fig.update_yaxes(title_text="ms", row=1, col=2)
fig.update_layout(height=460, legend=dict(orientation="h", y=-0.2), **LAYOUT_CLARO)
fig.show()

## 6.1 · Lendo a bancada

Três perguntas para discutir em sala.

1. No painel da esquerda, os pontos azuis (Dijkstra) crescem com a distância mais rápido que os verdes (A\*). Que forma geométrica explica isso? (Área do disco contra área da elipse.)
2. No painel da direita, a euclidiana em metros (roxo) é mais rápida que a great circle (verde) com praticamente os mesmos nós expandidos. De onde vem a diferença?
3. O guloso (laranja) é o mais rápido de todos e às vezes acerta o ótimo. Quando você aceitaria usá-lo? Pense em um jogo com milhares de agentes se movendo a cada quadro.

# 7 · Desafios

Três resolvidos, para estudar o código, e três abertos, para você.

## Desafio 1 · Marcos e a desigualdade triangular (resolvido)

A distância em linha reta é uma heurística **geométrica**. Ela ignora que a cidade tem um rio, uma lagoa e um parque de dunas sem ruas no meio. Os roteadores de verdade usam heurísticas **pré-computadas**. A mais elegante é a **ALT** (Goldberg e Harrelson, 2005), de *A\*, Landmarks and Triangle inequality*.

Escolha alguns nós como **marcos** (landmarks) $L$ e calcule, com Dijkstra, a distância de cada marco para todos os nós e de todos os nós para cada marco (em grafo dirigido são números diferentes). Pela desigualdade triangular, para qualquer marco,

$$d(u, t) \;\ge\; d(L, t) - d(L, u) \qquad\text{e}\qquad d(u, t) \;\ge\; d(u, L) - d(t, L).$$

O máximo dessas quantidades sobre todos os marcos é um limite inferior de $d(u, t)$, logo uma heurística admissível e consistente. Marcos nas bordas da cidade funcionam melhor, e a seleção gulosa abaixo (cada novo marco é o nó mais distante dos já escolhidos) é a receita clássica. O pré-processamento custa alguns Dijkstras completos e é feito uma única vez. Depois, cada consulta expande bem menos nós do que com a heurística geométrica, e a tabela de comparação mostra quanto.

In [ ]:
def escolher_landmarks(G, k=6, peso="length", semente=0):
    """Escolha gulosa de marcos (landmarks): começa em um nó aleatório e, a cada passo, adiciona
    o nó mais distante do conjunto já escolhido. Marcos nas bordas funcionam melhor."""
    rng = random.Random(semente)
    marcos = [rng.choice(list(G.nodes))]
    dist_min = nx.single_source_dijkstra_path_length(G, marcos[0], weight=peso)
    for _ in range(k - 1):
        novo = max(dist_min, key=dist_min.get)
        marcos.append(novo)
        d_novo = nx.single_source_dijkstra_path_length(G, novo, weight=peso)
        dist_min = {n: min(dist_min.get(n, math.inf), d_novo.get(n, math.inf)) for n in G.nodes}
    return marcos


def preparar_alt(G, marcos, peso="length"):
    """Pré-calcula, para cada marco L, as distâncias de L para todos (d_de) e de todos para L (d_para).
    Em um grafo dirigido os dois sentidos são diferentes."""
    G_rev = G.reverse(copy=False)
    d_de = {L: nx.single_source_dijkstra_path_length(G, L, weight=peso) for L in marcos}
    d_para = {L: nx.single_source_dijkstra_path_length(G_rev, L, weight=peso) for L in marcos}
    return d_de, d_para


def h_alt(d_de, d_para):
    """Heurística ALT. Pela desigualdade triangular, para qualquer marco L,
    d(u, t) >= d(L, t) - d(L, u)  e  d(u, t) >= d(u, L) - d(t, L).
    O máximo sobre os marcos continua sendo um limite inferior, logo é admissível (e consistente)."""
    def h(u, t):
        melhor = 0.0
        for L in d_de:
            a = d_de[L].get(t, math.inf) - d_de[L].get(u, math.inf)
            b = d_para[L].get(u, math.inf) - d_para[L].get(t, math.inf)
            for cand in (a, b):
                if cand == cand and cand > melhor and cand != math.inf:   # cand == cand descarta NaN (inf - inf)
                    melhor = cand
        return melhor
    return h

t0 = time.perf_counter()
MARCOS = escolher_landmarks(G, k=6)
D_DE, D_PARA = preparar_alt(G, MARCOS)
print(f"pré-processamento ALT com {len(MARCOS)} marcos: {time.perf_counter() - t0:.1f} s")
h_landmarks = h_alt(D_DE, D_PARA)

comparacao = []
for no_o, no_d in pares:
    o, d = NOS[no_o], NOS[no_d]
    b_gc = a_star(G, o, d, heuristica=h_gc, peso="length")
    b_alt = a_star(G, o, d, heuristica=h_landmarks, peso="length")
    assert math.isclose(b_gc.custo, b_alt.custo), "ALT tem que preservar o ótimo"
    comparacao.append(dict(par=f"{no_o.split(' (')[0]} → {no_d.split(' (')[0]}",
                           **{"A* great circle": b_gc.n_expandidos, "A* ALT": b_alt.n_expandidos}))
df_alt = pd.DataFrame(comparacao).set_index("par")
fig = go.Figure()
fig.add_bar(x=df_alt.index, y=df_alt["A* great circle"], name="A* great circle", marker_color=COR["A*"])
fig.add_bar(x=df_alt.index, y=df_alt["A* ALT"], name="A* ALT (6 marcos)", marker_color=COR["A* ponderado"])
fig.update_layout(barmode="group", title="Nós expandidos · heurística geométrica × heurística ALT",
                  height=460, xaxis_tickangle=-30, **LAYOUT_CLARO)
fig.show()
print(f"redução média de nós expandidos com ALT: {100 * (1 - (df_alt['A* ALT'] / df_alt['A* great circle']).mean()):.0f}%")

## Desafio 2 · Hora do rush (resolvido)

E se as avenidas ficarem três vezes mais lentas? Multiplicamos `travel_time` nas vias das classes `primary`, `secondary` e `trunk`, e rodamos o A\* de tempo de novo. Duas observações. A heurística de tempo **continua admissível**, porque as vias só ficaram mais lentas (o limite inferior continua válido). E a rota mais rápida muda de cara, desviando das avenidas. É o mecanismo básico de um aplicativo de navegação com trânsito em tempo real. O grafo muda, o algoritmo não.

In [ ]:
# Hora do rush sintética: as vias principais ficam 3x mais lentas.
G_rush = G.copy()
vias_lentas = {"primary", "primary_link", "trunk", "trunk_link", "secondary", "secondary_link"}
n_afetadas = 0
for u, v, k, dados in G_rush.edges(keys=True, data=True):
    tipo = dados.get("highway")
    tipos = set(tipo) if isinstance(tipo, list) else {tipo}
    if tipos & vias_lentas:
        dados["travel_time"] *= 3
        n_afetadas += 1
print(f"{n_afetadas:,} arestas ficaram 3x mais lentas")

# A heurística de tempo continua admissível: as vias só ficaram mais lentas, nunca mais rápidas.
h_t = h_tempo(G, V_MAX_KPH)
rota_normal = a_star(G, ORIGEM, DESTINO, heuristica=h_t, peso="travel_time").caminho
rota_rush = a_star(G_rush, ORIGEM, DESTINO, heuristica=h_t, peso="travel_time").caminho
for nome, grafo, rota in [("sem trânsito", G, rota_normal), ("hora do rush", G_rush, rota_rush)]:
    dist_m, tempo_s = resumo_rota(grafo, rota)
    print(f"{nome:14s} {dist_m/1000:5.1f} km · {tempo_s/60:5.1f} min")
rota_sobre_mapa(G, [rota_normal, rota_rush], nomes=["mais rápida sem trânsito", "mais rápida na hora do rush"],
                cores=[COR["A* ponderado"], COR["Guloso"]]).show()

## Desafio 3 · Outra cidade (resolvido)

Nada no código é específico de Natal. A função `carregar_grafo` aceita qualquer lugar que o OpenStreetMap conheça. A célula abaixo baixa Mossoró e repete a pegada de busca para um par aleatório. Ela vem desligada para o notebook rodar rápido. Troque o interruptor e experimente com a sua cidade.

In [ ]:
EXECUTAR_DESAFIO_3 = False   # mude para True para baixar outra cidade (cerca de um minuto)

if EXECUTAR_DESAFIO_3:
    G_outra = carregar_grafo("Mossoró, Rio Grande do Norte, Brazil", "mossoro_drive.graphml", url=None)
    nos_outra = list(G_outra.nodes)
    rng_cidade = random.Random(1)
    o2, d2 = rng_cidade.sample(nos_outra, 2)
    h2 = h_grande_circulo(G_outra)
    buscas_outra = {"Dijkstra": a_star(G_outra, o2, d2, peso="length"),
                    "A*": a_star(G_outra, o2, d2, heuristica=h2, peso="length")}
    desenhar_pegada(G_outra, buscas_outra, tamanho_ponto=4, titulo="Mossoró · mesmo código, outra cidade")
    plt.show()
else:
    print("Desafio 3 desligado. Troque EXECUTAR_DESAFIO_3 para True e rode de novo.")

## Desafio 4 · A\* bidirecional (aberto)

Rode duas buscas ao mesmo tempo, uma da origem para o destino e outra do destino para a origem (no grafo reverso), alternando expansões. Em vez de uma elipse, a pegada vira **duas elipses menores** que se tocam no meio, e a área total cai. A dificuldade está no critério de parada. Quando as duas fronteiras se encontram em um nó, o caminho que passa por ele **não é necessariamente o ótimo**, e a busca precisa continuar até que a soma dos menores $f$ das duas filas supere o melhor caminho encontrado. Implemente, valide contra `nx.bidirectional_dijkstra` e compare a pegada com a do A\* comum nos doze pares da bancada.

## Desafio 5 · Quebre o A\* (aberto)

Escolha dez pares aleatórios de nós de Natal. Para $w \in \{1{,}0;\ 1{,}1;\ 1{,}25;\ 1{,}5;\ 2;\ 3;\ 5\}$ rode o A\* ponderado com a heurística great circle e registre, para cada par, os nós expandidos e a razão custo/ótimo. Desenhe as duas curvas em função de $w$. Qual é o maior $w$ que ainda devolve o ótimo em todos os dez pares? Compare a razão observada com o limite teórico $w$. Por que a distância entre a curva e o limite é tão grande?

## Desafio 6 · Terreno com custo (aberto)

Volte ao tabuleiro e dê a cada célula um custo de 1 a 5 (areia, asfalto, lama). O peso da aresta $(u, v)$ passa a ser o custo da célula de chegada $v$. Qual é a heurística admissível mais apertada que você consegue escrever? (Dica. Manhattan vezes o menor custo do tabuleiro é admissível. Dá para fazer melhor?) Mostre, com uma animação, o A\* contornando a lama.

In [ ]:
# ---- Desafio 4: seu código aqui ----
# Sugestão de assinatura:
# def a_star_bidirecional(G, origem, destino, heuristica, peso="length"):
#     ...
#     return caminho, custo, n_expandidos

In [ ]:
# ---- Desafio 5: seu código aqui ----
# Dica: reutilize `bancada` com configurações ("A* w=1.1", h_gc, "length", G, 1.1), ("A* w=1.25", ...), ...
# e desenhe custo_vs_otimo e nos_expandidos em função de w.

In [ ]:
# ---- Desafio 6: seu código aqui ----
# Dica: comece de criar_tabuleiro(), sorteie um custo por célula (1 a 5) e defina o peso da aresta (u, v)
# como o custo da célula de chegada v. Qual é a heurística admissível mais apertada que você consegue?

# 8 · Quiz relâmpago, checklist e referências

Responda antes de abrir.

<details><summary><b>1.</b> O A* descobre o destino no passo 120 e o expande no passo 157. Em que passo ele pode parar com garantia de ótimo?</summary>
No passo 157. Descobrir o destino só significa que existe <i>algum</i> caminho até ele. Enquanto ele não for o nó de menor $f$ da fronteira, pode haver um caminho mais barato por outro nó ainda aberto.
</details>

<details><summary><b>2.</b> Uma heurística é admissível e inconsistente. O A* com conjunto fechado sem reabertura ainda acha o ótimo?</summary>
Não necessariamente. Um nó pode ser fechado com um $g$ que não é o ótimo e nunca mais ser revisitado. Com reabertura (<code>reabrir=True</code>) o ótimo volta a ser garantido, ao custo de mais expansões.
</details>

<details><summary><b>3.</b> Por que a euclidiana em graus produziu exatamente o mesmo número de nós expandidos que o Dijkstra, e não apenas um número parecido?</summary>
Porque $h$ em graus (ordem de $10^{-1}$) é menor que a menor diferença entre dois valores distintos de $g$ em metros. A ordem da fila fica idêntica à do Dijkstra em todos os passos. É uma heurística admissível e consistente, só que invisível para a fila.
</details>

<details><summary><b>4.</b> A heurística de tempo assume 40 km/h em uma cidade cujo limite máximo é 80 km/h. O que você pode garantir sobre a rota devolvida?</summary>
Que o tempo dela é no máximo $80/40 = 2$ vezes o tempo ótimo. É o A* ponderado com $w = 2$. Na prática a razão observada fica bem abaixo disso.
</details>

<details><summary><b>5.</b> Um A* expande metade dos nós do Dijkstra e demora o mesmo tempo. Dê duas causas possíveis.</summary>
O custo de cada chamada da heurística (trigonometria em Python) e o acesso aos atributos dos nós a cada chamada. Pré-calcular as coordenadas em um dicionário e trocar great circle por euclidiana no grafo projetado atacam as duas causas.
</details>

### ✅ Checklist de saída

- [ ] Sei escrever o A\* com `heapq` e explicar remoção preguiçosa, contador de desempate e parada ao expandir.
- [ ] Sei definir heurística admissível e consistente e sei qual garantia cada uma dá.
- [ ] Sei prever o formato da pegada de busca de Dijkstra, A\* e guloso e explicar a elipse.
- [ ] Sei enunciar o limite $w \cdot C^*$ do A\* ponderado e traduzi-lo para "velocidade assumida".
- [ ] Sei diagnosticar heurística em unidade errada olhando só a tabela de resultados.
- [ ] Sei por que nós expandidos e tempo de parede medem coisas diferentes.

### 📚 Referências

- Hart, P. E., Nilsson, N. J. e Raphael, B. (1968). *A Formal Basis for the Heuristic Determination of Minimum Cost Paths*. IEEE Transactions on Systems Science and Cybernetics, 4(2), 100–107. A correção de 1972 (SIGART Newsletter, 37) introduz a condição de consistência.
- Pohl, I. (1970). *Heuristic search viewed as path finding in a graph*. Artificial Intelligence, 1(3–4), 193–204. O A\* ponderado.
- Goldberg, A. V. e Harrelson, C. (2005). *Computing the shortest path: A\* search meets graph theory*. SODA 2005. A heurística ALT.
- Russell, S. e Norvig, P. *Artificial Intelligence: A Modern Approach*, capítulo de busca informada. Provas de otimalidade e fator de ramificação efetivo.
- Patel, A. *Introduction to the A\* Algorithm* e *Implementation of A\**, Red Blob Games (redblobgames.com). A melhor explicação visual que existe, e a inspiração do tabuleiro deste notebook.
- Boeing, G. (2017). *OSMnx: New methods for acquiring, constructing, analyzing, and visualizing complex street networks*. Computers, Environment and Urban Systems, 65, 126–139. E Boeing, G. (2025). *Modeling and Analyzing Urban Networks and Amenities with OSMnx*. Geographical Analysis.
- Documentação do NetworkX, `nx.astar_path` (a implementação de referência usada na bancada, que também usa `heapq` com remoção preguiçosa).
- honzaap, *Pathfinding* (github.com/honzaap/Pathfinding). Visualizador interativo de Dijkstra e A\* em mapas reais, mostrado nos slides.